# 01 - Data Cleaning and Preparation

**Kenya Food Price Inflation Tracker**

## Objective
Load and clean the WFP Kenya food price dataset, preparing it for analysis.

## Steps
1. Load raw WFP data
2. Create date column from year/month
3. Handle missing values and outliers
4. Filter to core staple commodities
5. Create cleaned datasets

## Outputs
- `data/clean/wfp_kenya_clean.csv`
- `data/clean/wfp_core_staples.csv`
- `data/clean/wfp_monthly_avg.csv`

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print('✓ Libraries imported')

✓ Libraries imported


## 1. Load Raw Data

In [13]:
# Load data
try:
    df = pd.read_csv('../data/raw/wfp_food_prices_kenya_full.csv')
except FileNotFoundError:
    df = pd.read_csv('data/raw/wfp_food_prices_kenya_full.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nDate range: {df["mp_year"].min()}-{df["mp_month"].min():02d} to {df["mp_year"].max()}-{df["mp_month"].max():02d}')
df.head()

Dataset shape: (8884, 18)

Columns: ['adm0_id', 'adm0_name', 'adm1_id', 'adm1_name', 'mkt_id', 'mkt_name', 'cm_id', 'cm_name', 'cur_id', 'cur_name', 'pt_id', 'pt_name', 'um_id', 'um_name', 'mp_month', 'mp_year', 'mp_price', 'mp_commoditysource']

Date range: 2006-01 to 2021-12


,adm0_id,adm0_name,adm1_id,adm1_name,mkt_id,mkt_name,cm_id,cm_name,cur_id,cur_name,pt_id,pt_name,um_id,um_name,mp_month,mp_year,mp_price,mp_commoditysource
0,133.0,Kenya,51326,Coast,191,Mombasa,50,Beans - Wholesale,0.0,KES,14,Wholesale,5,KG,1,2006,33.630,NaN
1,133.0,Kenya,51326,Coast,191,Mombasa,50,Beans - Wholesale,0.0,KES,14,Wholesale,5,KG,2,2006,39.478,NaN
2,133.0,Kenya,51326,Coast,191,Mombasa,50,Beans - Wholesale,0.0,KES,14,Wholesale,5,KG,3,2006,44.686,NaN
3,133.0,Kenya,51326,Coast,191,Mombasa,50,Beans - Wholesale,0.0,KES,14,Wholesale,5,KG,4,2006,43.837,NaN
4,133.0,Kenya,51326,Coast,191,Mombasa,50,Beans - Wholesale,0.0,KES,14,Wholesale,5,KG,5,2006,39.734,NaN


In [14]:
# Check data types and missing values
print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])

print('\n=== Data Types ===')
print(df.dtypes)


=== Missing Values ===
mp_commoditysource    8884
dtype: int64

=== Data Types ===
adm0_id               float64
adm0_name              object
adm1_id                 int64
adm1_name              object
mkt_id                  int64
mkt_name               object
cm_id                   int64
cm_name                object
cur_id                float64
cur_name               object
pt_id                   int64
pt_name                object
um_id                   int64
um_name                object
mp_month                int64
mp_year                 int64
mp_price              float64
mp_commoditysource    float64
dtype: object


## 2. Create Date Column

In [15]:
# Create date column from year and month
df['date'] = pd.to_datetime({'year': df['mp_year'], 'month': df['mp_month'], 'day': 1})
df = df.sort_values('date').reset_index(drop=True)

print(f'✓ Date column created')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
df[['date', 'cm_name', 'mp_price', 'adm1_name', 'mkt_name']].head()

✓ Date column created
Date range: 2006-01-01 00:00:00 to 2021-08-01 00:00:00


,date,cm_name,mp_price,adm1_name,mkt_name
0,2006-01-01,Beans - Wholesale,33.63,Coast,Mombasa
1,2006-01-01,Beans (dry) - Wholesale,3246.00,Coast,Mombasa
2,2006-01-01,Maize (white) - Retail,26.00,Rift Valley,Lodwar (Turkana)
3,2006-01-01,Beans (dry) - Wholesale,2799.00,Rift Valley,Eldoret town
4,2006-01-01,Sorghum - Wholesale,1800.00,Eastern,Kitui


## 3. Data Cleaning

In [16]:
# Drop rows with missing prices
print(f'Rows before: {len(df)}')
df = df.dropna(subset=['mp_price'])
df = df[df['mp_price'] > 0]
print(f'Rows after: {len(df)}')
print(f'Removed: {len(df) - len(df)} invalid price records')

Rows before: 8884
Rows after: 8884
Removed: 0 invalid price records


In [17]:
# Remove outliers using IQR method (per commodity)
def remove_outliers_iqr(group, column='mp_price', multiplier=3.0):
    Q1 = group[column].quantile(0.25)
    Q3 = group[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return group[(group[column] >= lower) & (group[column] <= upper)]

print(f'Before outlier removal: {len(df):,} rows')
df_clean = df.groupby('cm_name', group_keys=False).apply(remove_outliers_iqr)
print(f'After outlier removal: {len(df_clean):,} rows')
print(f'Removed: {len(df) - len(df_clean):,} outlier records')

Before outlier removal: 8,884 rows
After outlier removal: 8,856 rows
Removed: 28 outlier records


## 4. Focus on Core Staple Commodities

In [18]:
# Check available commodities
print('All commodities:')
print(df_clean['cm_name'].value_counts())

All commodities:
cm_name
Maize (white) - Retail              1052
Maize - Wholesale                    832
Beans (dry) - Wholesale              733
Maize (white) - Wholesale            733
Beans - Wholesale                    724
Sorghum - Wholesale                  708
Potatoes (Irish) - Wholesale         706
Beans (dry) - Retail                 556
Oil (vegetable) - Retail             341
Milk (cow, pasteurized) - Retail     178
Bread - Retail                       178
Sugar - Retail                       165
Maize flour - Retail                 164
Wheat flour - Retail                 164
Rice - Retail                        160
Salt - Retail                        158
Milk (UHT) - Retail                  153
Maize - Retail                       152
Bananas - Retail                     150
Potatoes (Irish) - Retail            145
Sorghum - Retail                     109
Meat (beef) - Retail                 101
Meat (goat) - Retail                  81
Fuel (kerosene) - Retail        

In [19]:
# Define core staples (using ACTUAL commodity names from data)
# Look for: Maize, Beans, Rice, Oil, Sugar, Milk
core_staples = [
    'Maize - Retail',
    'Maize - Wholesale', 
    'Maize (white) - Retail',
    'Beans (dry) - Retail',
    'Beans - Wholesale',
    'Rice - Retail',
    'Oil (vegetable) - Retail',
    'Sugar - Retail',
    'Milk (cow, pasteurized) - Retail'
]

# Filter to commodities that actually exist in data
available_staples = [c for c in core_staples if c in df_clean['cm_name'].values]
print(f'\nCore staples available: {len(available_staples)}')
print(available_staples)

df_staples = df_clean[df_clean['cm_name'].isin(available_staples)].copy()
print(f'\nStaples dataset: {df_staples.shape}')
print(df_staples['cm_name'].value_counts())


Core staples available: 9
['Maize - Retail', 'Maize - Wholesale', 'Maize (white) - Retail', 'Beans (dry) - Retail', 'Beans - Wholesale', 'Rice - Retail', 'Oil (vegetable) - Retail', 'Sugar - Retail', 'Milk (cow, pasteurized) - Retail']

Staples dataset: (4160, 19)
cm_name
Maize (white) - Retail              1052
Maize - Wholesale                    832
Beans - Wholesale                    724
Beans (dry) - Retail                 556
Oil (vegetable) - Retail             341
Milk (cow, pasteurized) - Retail     178
Sugar - Retail                       165
Rice - Retail                        160
Maize - Retail                       152
Name: count, dtype: int64


## 5. Create Monthly Aggregated Data

In [20]:
# Aggregate to monthly averages (by commodity, region, and market)
df_monthly = df_staples.groupby(
    ['date', 'cm_name', 'adm1_name', 'mkt_name']
).agg({
    'mp_price': ['mean', 'std', 'count'],
    'mkt_id': 'first',
    'cur_name': 'first'
}).reset_index()

# Flatten column names
df_monthly.columns = ['date', 'commodity', 'region', 'market', 
                       'price_mean', 'price_std', 'obs_count',
                       'market_id', 'currency']

print(f'Monthly dataset: {df_monthly.shape}')
df_monthly.head(10)

Monthly dataset: (4160, 9)


,date,commodity,region,market,price_mean,price_std,obs_count,market_id,currency
0,2006-01-01,Beans (dry) - Retail,Eastern,Kitui,39.000,NaN,1,187,KES
1,2006-01-01,Beans - Wholesale,Coast,Mombasa,33.630,NaN,1,191,KES
2,2006-01-01,Beans - Wholesale,Nairobi,Nairobi,42.311,NaN,1,184,KES
3,2006-01-01,Beans - Wholesale,Nyanza,Kisumu,39.608,NaN,1,186,KES
4,2006-01-01,Beans - Wholesale,Rift Valley,Eldoret town,45.854,NaN,1,185,KES
5,2006-01-01,Maize (white) - Retail,Eastern,Kitui,17.000,NaN,1,187,KES
6,2006-01-01,Maize (white) - Retail,Eastern,Marsabit,21.000,NaN,1,190,KES
7,2006-01-01,Maize (white) - Retail,North Eastern,Mandera,30.000,NaN,1,189,KES
8,2006-01-01,Maize (white) - Retail,Rift Valley,Lodwar (Turkana),26.000,NaN,1,188,KES
9,2006-01-01,Maize - Wholesale,Coast,Mombasa,16.131,NaN,1,191,KES


## 6. Save Cleaned Datasets

In [21]:
# Create output directory
Path('../data/clean').mkdir(parents=True, exist_ok=True)

# Save datasets
df_clean.to_csv('../data/clean/wfp_kenya_clean.csv', index=False)
print(f'✓ Saved: wfp_kenya_clean.csv ({len(df_clean):,} rows)')

df_staples.to_csv('../data/clean/wfp_core_staples.csv', index=False)
print(f'✓ Saved: wfp_core_staples.csv ({len(df_staples):,} rows)')

df_monthly.to_csv('../data/clean/wfp_monthly_avg.csv', index=False)
print(f'✓ Saved: wfp_monthly_avg.csv ({len(df_monthly):,} rows)')

✓ Saved: wfp_kenya_clean.csv (8,856 rows)
✓ Saved: wfp_core_staples.csv (4,160 rows)
✓ Saved: wfp_monthly_avg.csv (4,160 rows)


## 7. Summary

In [22]:
print('=== DATA CLEANING SUMMARY ===')
print(f'Original records: {len(df):,}')
print(f'After cleaning: {len(df_clean):,}')
print(f'Core staples: {len(df_staples):,}')
print(f'Monthly aggregated: {len(df_monthly):,}')
print(f'\nDate range: {df_staples["date"].min()} to {df_staples["date"].max()}')
print(f'Commodities: {df_staples["cm_name"].nunique()}')
print(f'Markets: {df_staples["mkt_name"].nunique()}')
print(f'Regions: {df_staples["adm1_name"].nunique()}')
print('\n✅ Data cleaning complete!')

=== DATA CLEANING SUMMARY ===
Original records: 8,884
After cleaning: 8,856
Core staples: 4,160
Monthly aggregated: 4,160

Date range: 2006-01-01 00:00:00 to 2021-08-01 00:00:00
Commodities: 9
Markets: 45
Regions: 6

✅ Data cleaning complete!
